In [1]:
# Parsing the text labels and verifying the text path

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from pathlib import Path
import pandas as pd
from collections import Counter
import editdistance
import numpy as np
from tqdm import tqdm
import random

# Kaggle input path
BASE_INPUT = Path("/kaggle/input/datasets/nibinv23/iam-handwriting-word-database/iam_words")
WORDS_TXT = BASE_INPUT / "words.txt"
WORDS_IMG_DIR = BASE_INPUT / "words"   

TARGET_HEIGHT = 32
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
EPOCHS = 30
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Below code is used store the id and text in a python list as each tuple has text id and the word.
def parse_iam_words(words_txt_path, words_img_dir):
    records = []
    with open(WORDS_TXT, "r") as f:
        for line in f:
            line = line.strip() # used to clean a string removes spaces and newline
            if not line or line.startswith("#"):
                continue
            parts = line.split() # breaks the line into list of words using whitespace stores in a list named parts.
            if len(parts) < 9:               # The size of parts array is less than 9 then skip the line.
                continue
            word_id = parts[0]               # e.g., a01-000u-00-00
            seg_status = parts[1]            # ok or err
            transcription = " ".join(parts[8:])  # transcription may contain spaces
    
            # keep only well‑segmented words
            if seg_status != "ok":
                continue
            
             # Build expected image path
            folder1, folder2, *_ = word_id.split("-")
            img_path = words_img_dir / folder1 / f"{folder1}-{folder2}" / f"{word_id}.png"

            try:
                with Image.open(img_path) as img:
                    img.verify()          # lightweight validity check
                records.append((word_id, transcription, str(img_path)))
            except Exception:
                # Corrupt file skip it silently
                pass

            
    return  pd.DataFrame(records, columns=["word_id", "text", "img_path"])

# Call the function
df = parse_iam_words(WORDS_TXT, WORDS_IMG_DIR)

print(f"Total valid samples: {len(df)}")
df.head()

Total valid samples: 38304


,word_id,text,img_path
0,a01-000u-00-00,A,/kaggle/input/datasets/nibinv23/iam-handwritin...
1,a01-000u-00-01,MOVE,/kaggle/input/datasets/nibinv23/iam-handwritin...
2,a01-000u-00-02,to,/kaggle/input/datasets/nibinv23/iam-handwritin...
3,a01-000u-00-03,stop,/kaggle/input/datasets/nibinv23/iam-handwritin...
4,a01-000u-00-04,Mr.,/kaggle/input/datasets/nibinv23/iam-handwritin...


In [2]:
# training validation test split:
from sklearn.model_selection import train_test_split

train_df, rest_df = train_test_split(df, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(rest_df, test_size=0.5, random_state=42)
print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

Train: 30643, Val: 3830, Test: 3831


In [3]:
# character set and mapping

from collections import Counter

all_text = ''.join(df['text'].values)
char_counts = Counter(all_text)
chars = sorted(char_counts.keys())

# Add a blank token for CTC (index 0)
chars = ['<blank>'] + chars
char2idx = {c: i for i, c in enumerate(chars)}
idx2char = {i: c for c, i in char2idx.items()}
num_classes = len(chars)

print(f"Number of unique characters: {num_classes}")
print(f"Character set: {chars}")

Number of unique characters: 78
Character set: ['<blank>', ' ', '!', '"', '#', "'", '(', ')', '*', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [4]:
import torch
from torch.utils.data import Dataset
from PIL import Image
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# dataset class 
class IAMWordsDataset(Dataset):
    def __init__(self, df, char2idx, target_height=32):
        self.df = df.reset_index(drop=True)
        self.char2idx = char2idx
        self.target_height = target_height
        # Transform: to tensor + standardise pixel values to [0,1]
        self.transform = transforms.ToTensor()

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["img_path"]).convert("L")  # grayscale
        # Resize keeping aspect ratio, height = target_height
        w, h = img.size
        new_w = int(self.target_height * (w / h))
        img = img.resize((new_w, self.target_height), Image.BICUBIC)
        img_tensor = self.transform(img)                # (1, H, W)
        # Encode label
        label = row["text"]
        label_enc = [self.char2idx[c] for c in label if c in self.char2idx]
        return img_tensor, torch.tensor(label_enc, dtype=torch.long), label, new_w

# collate function for various image width
def collate_fn(batch):
    images, labels, texts, widths = zip(*batch)
    max_width = max(img.shape[2] for img in images)
    padded_images = []
    for img in images:
        pad = max_width - img.shape[2]
        padded = F.pad(img, (0, pad), value=0)
        padded_images.append(padded)
    images_batch = torch.stack(padded_images, dim=0)   # (B, 1, H, Wmax)

    labels_padded = nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value=0)
    label_lengths = torch.tensor([len(l) for l in labels], dtype=torch.long)

    # For the CNN used, there is NO horizontal stride; output width = input width
    input_lengths = torch.tensor(widths, dtype=torch.long)

    return images_batch, labels_padded, label_lengths, input_lengths, texts

# dataloaders
dataset = IAMWordsDataset(df, char2idx, target_height=32)
dataloader = DataLoader(dataset, 
                        batch_size=32, 
                        shuffle=True,
                        collate_fn=collate_fn, 
                        num_workers=2
                       )

In [5]:
# residual block
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, 1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return self.relu(out)

In [6]:
class CRNN(nn.Module):
    def __init__(self, num_classes, img_height=32):
        super().__init__()
        self.cnn = nn.Sequential(
            # initial conv
            nn.Conv2d(1, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            # residual blocks with pooling (only vertical stride)
            ResidualBlock(64, 64),
            nn.MaxPool2d((2, 1)),   # H/2
            ResidualBlock(64, 128),
            nn.MaxPool2d((2, 1)),   # H/4
            ResidualBlock(128, 256),
            ResidualBlock(256, 256),
            nn.MaxPool2d((2, 1)),   # H/8
            ResidualBlock(256, 512),
            nn.MaxPool2d((2, 1)),   # H/16
            ResidualBlock(512, 512),
            nn.MaxPool2d((2, 1)),   # H/32 → height = 1
        )
        # Recurrent part
        self.rnn = nn.LSTM(512, 256, num_layers=2,
                           bidirectional=True, batch_first=True)
        self.fc = nn.Linear(512, num_classes)  # 256*2

    def forward(self, x):
        # x: (B, 1, H, W)
        feats = self.cnn(x)                    # (B, 512, 1, W')
        feats = feats.squeeze(2)               # (B, 512, W')
        feats = feats.permute(0, 2, 1)         # (B, W', 512) batch_first
        rnn_out, _ = self.rnn(feats)           # (B, W', 512)
        logits = self.fc(rnn_out)              # (B, W', num_classes)
        return F.log_softmax(logits, dim=2)

In [7]:
# greedy function

def greedy_decode(log_probs, idx2char, blank=0):
    """log_probs: (B, T, C) or (T, B, C)"""
    if log_probs.dim() == 3 and log_probs.shape[0] != log_probs.shape[1]:
        log_probs = log_probs.permute(1, 0, 2)  # ensure (T, B, C)
    _, max_idx = torch.max(log_probs, dim=2)    # (T, B)
    max_idx = max_idx.transpose(0, 1).cpu().numpy()  # (B, T)
    decoded = []
    for b in range(max_idx.shape[0]):
        seq = []
        prev = blank
        for idx in max_idx[b]:
            if idx != blank and idx != prev:
                seq.append(idx2char[idx])
            prev = idx
        decoded.append("".join(seq))
    return decoded

In [8]:
# calculate CER
def compute_cer(pred_texts, true_texts):
    total_errs = 0
    total_chars = 0
    for pred, true in zip(pred_texts, true_texts):
        total_errs += editdistance.eval(pred, true)
        total_chars += len(true)
    if total_chars == 0:
        return float('inf')
    return total_errs / total_chars

def compute_wer(pred_texts, true_texts):
    total_errs = 0
    total_words = 0
    for pred, true in zip(pred_texts, true_texts):
        pred_words = pred.split()
        true_words = true.split()
        total_errs += editdistance.eval(pred_words, true_words)
        total_words += len(true_words)
    if total_words == 0:
        return float('inf')
    return total_errs / total_words

In [9]:
# training and validation loop
def train_one_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for batch in tqdm(dataloader, desc="Training", leave=False):
        images, labels, label_lengths, input_lengths, _ = batch
        images = images.to(device)
        labels = labels.to(device)
        label_lengths = label_lengths.to(device)
        input_lengths = input_lengths.to(device)

        optimizer.zero_grad()
        log_probs = model(images)                  # (B, T, C)
        # CTC loss expects (T, B, C)
        log_probs_ctc = log_probs.permute(1, 0, 2)
        loss = criterion(log_probs_ctc, labels, input_lengths, label_lengths)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(dataloader)

def validate(model, dataloader, criterion, device, idx2char):
    model.eval()
    total_loss = 0
    all_preds = []
    all_trues = []
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Validation", leave=False):
            images, labels, label_lengths, input_lengths, texts = batch
            images = images.to(device)
            labels = labels.to(device)
            label_lengths = label_lengths.to(device)
            input_lengths = input_lengths.to(device)

            log_probs = model(images)
            log_probs_ctc = log_probs.permute(1, 0, 2)
            loss = criterion(log_probs_ctc, labels, input_lengths, label_lengths)
            total_loss += loss.item()
            pred_texts = greedy_decode(log_probs, idx2char)
            all_preds.extend(pred_texts)
            all_trues.extend(texts)
    cer = compute_cer(all_preds, all_trues)
    wer = compute_wer(all_preds, all_trues)
    return total_loss / len(dataloader), cer, wer, all_preds, all_trues


In [10]:
# full training script
train_dataset = IAMWordsDataset(train_df, char2idx)
val_dataset   = IAMWordsDataset(val_df, char2idx)
test_dataset  = IAMWordsDataset(test_df, char2idx)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate_fn, num_workers=2)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collate_fn, num_workers=2)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collate_fn, num_workers=2)

model = CRNN(num_classes).to(DEVICE)
criterion = nn.CTCLoss(blank=0, zero_infinity=True)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

best_val_cer = float('inf')
best_model_path = "best_crnn_iam.pth"

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
    val_loss, val_cer, val_wer, _, _ = validate(model, val_loader, criterion, DEVICE, idx2char)

    print(f"Epoch {epoch:02d} | Train Loss: {train_loss:.4f} | "
          f"Val Loss: {val_loss:.4f} | CER: {val_cer:.4f} | WER: {val_wer:.4f}")

    if val_cer < best_val_cer:
        best_val_cer = val_cer
        torch.save(model.state_dict(), best_model_path)
        print("  -> Best model saved (CER improved)")

print("Training finished.")

Epoch 01 | Train Loss: 3.0909 | Val Loss: 2.6125 | CER: 0.8207 | WER: 0.8648
  -> Best model saved (CER improved)


Epoch 02 | Train Loss: 2.2895 | Val Loss: 1.9937 | CER: 0.7415 | WER: 0.7515
  -> Best model saved (CER improved)


Epoch 03 | Train Loss: 1.7728 | Val Loss: 1.5157 | CER: 0.6003 | WER: 0.7010
  -> Best model saved (CER improved)


Epoch 04 | Train Loss: 1.1941 | Val Loss: 0.9031 | CER: 0.3104 | WER: 0.5527
  -> Best model saved (CER improved)


Epoch 05 | Train Loss: 0.8223 | Val Loss: 0.7213 | CER: 0.2413 | WER: 0.4809
  -> Best model saved (CER improved)


...

Epoch 26 | Train Loss: 0.0897 | Val Loss: 0.3750 | CER: 0.0993 | WER: 0.2636
  -> Best model saved (CER improved)


Epoch 27 | Train Loss: 0.0761 | Val Loss: 0.3846 | CER: 0.1033 | WER: 0.2696


Epoch 28 | Train Loss: 0.0831 | Val Loss: 0.3701 | CER: 0.0981 | WER: 0.2620
  -> Best model saved (CER improved)


Epoch 29 | Train Loss: 0.0775 | Val Loss: 0.3777 | CER: 0.0998 | WER: 0.2615


Epoch 30 | Train Loss: 0.0609 | Val Loss: 0.3982 | CER: 0.0985 | WER: 0.2618
Training finished.


In [11]:
# testing script
print("\nLoading best model for testing...")
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
test_loss, test_cer, test_wer, test_preds, test_trues = validate(
    model, test_loader, criterion, DEVICE, idx2char
)
print(f"Test Loss: {test_loss:.4f} | CER: {test_cer:.4f} | WER: {test_wer:.4f}")

# Show some example predictions
print("\nSample predictions:")
for i in range(5):
    idx = random.randint(0, len(test_preds)-1)
    print(f"  True: '{test_trues[idx]}'  |  Pred: '{test_preds[idx]}'")


Loading best model for testing...


Test Loss: 0.4104 | CER: 0.1059 | WER: 0.2742

Sample predictions:
  True: '.'  |  Pred: '.'
  True: 'corresponding'  |  Pred: 'correspouding'
  True: 'composition'  |  Pred: 'composition'
  True: '.'  |  Pred: '.'
  True: 'we'  |  Pred: 'us'
